# Лабораторная работа 2+3: Классификация + Оптимизация гиперпараметров

**Задача:** Предсказание одобрения кредита (LoanApproved)

**Основная метрика:** ROC-AUC > 0.75

**Дополнительные метрики:** Precision, Recall, F1-score, PR-AUC

## 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, auc,
    precision_score, recall_score, f1_score
)
from lightgbm import LGBMClassifier
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Загрузка данных

In [ ]:
train_df = pd.read_csv('train_c.csv')
test_df = pd.read_csv('test_c.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()

## 3. Анализ данных

In [ ]:
print('Информация о данных:')
print(f'Пропущенные значения: {train_df.isnull().sum().sum()}')
print(f'\nРаспределение целевой переменной:')
print(train_df['LoanApproved'].value_counts())
print(f'\nБаланс классов:')
print(train_df['LoanApproved'].value_counts(normalize=True).round(3))

### 3.1 Распределение целевой переменной

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df['LoanApproved'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Распределение', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(['Отклонен (0)', 'Одобрен (1)'], rotation=0)

train_df['LoanApproved'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4'])
axes[1].set_title('Доля классов', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Подготовка данных

In [ ]:
test_ids = test_df['ID'].valuesif 'ID' in test_df.columns:    test_df = test_df.drop('ID', axis=1)X = train_df.drop('LoanApproved', axis=1)y = train_df['LoanApproved']mask = ~y.isna()X = X[mask].reset_index(drop=True)y = y[mask].reset_index(drop=True)print(f'Данные после очистки: {X.shape}')

### 4.1 Обработка признаков

In [ ]:
# Обработка датfor df in [X, test_df]:    if 'ApplicationDate' in df.columns:        df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'], errors='coerce')        df['ApplicationYear'] = df['ApplicationDate'].dt.year        df['ApplicationMonth'] = df['ApplicationDate'].dt.month        df.drop('ApplicationDate', axis=1, inplace=True)# Категориальные признакиcategorical_features = ['MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose', 'EmploymentStatus', 'EducationLevel']label_encoders = {}for col in categorical_features:    if col in X.columns:        le = LabelEncoder()        combined = pd.concat([X[col].astype(str), test_df[col].astype(str)])        le.fit(combined)        X[col] = le.transform(X[col].astype(str))        test_df[col] = le.transform(test_df[col].astype(str))        label_encoders[col] = le# Заполнение пропусковnumeric_features = X.select_dtypes(include=[np.number]).columns.tolist()imputer = SimpleImputer(strategy='median')X[numeric_features] = imputer.fit_transform(X[numeric_features])test_df[numeric_features] = imputer.transform(test_df[numeric_features])# Новые признакиfor df in [X, test_df]:    df['Income_to_Loan'] = df['AnnualIncome'] / (df['LoanAmount'] + 1)    df['Debt_to_Income'] = df['MonthlyDebtPayments'] / (df['MonthlyIncome'] + 1)    df['Assets_to_Liabilities'] = df['TotalAssets'] / (df['TotalLiabilities'] + 1)    df['Credit_Utilization_Score'] = df['CreditScore'] * (1 - df['CreditCardUtilizationRate'])    df['Age_Income'] = df['Age'] * df['AnnualIncome']    df.replace([np.inf, -np.inf], np.nan, inplace=True)    df.fillna(df.median(), inplace=True)print(f'Итого признаков: {X.shape[1]}')

## 5. Разделение и масштабирование

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print(f'Train: {X_train.shape}, Val: {X_val.shape}')scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_val_scaled = scaler.transform(X_val)X_test_scaled = scaler.transform(test_df)

## 6. Оптимизация гиперпараметров (Optuna)

In [ ]:
def objective(trial):    params = {        'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'boosting_type': 'gbdt',        'n_estimators': trial.suggest_int('n_estimators', 100, 500),        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),        'num_leaves': trial.suggest_int('num_leaves', 20, 150),        'max_depth': trial.suggest_int('max_depth', 3, 15),        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),        'subsample': trial.suggest_float('subsample', 0.5, 1.0),        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),        'random_state': 42    }    model = LGBMClassifier(**params)    model.fit(X_train, y_train)    y_pred_proba = model.predict_proba(X_val)[:, 1]    return roc_auc_score(y_val, y_pred_proba)study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))study.optimize(objective, n_trials=50, show_progress_bar=True)print(f'\nЛучший ROC-AUC: {study.best_value:.4f}')

## 7. Обучение финальной модели

In [ ]:
best_params = study.best_params.copy()best_params.update({'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'boosting_type': 'gbdt', 'random_state': 42})final_model = LGBMClassifier(**best_params)final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc')y_pred_proba = final_model.predict_proba(X_val)[:, 1]y_pred = final_model.predict(X_val)

## 8. Оценка модели - ВСЕ МЕТРИКИ

In [ ]:
roc_auc = roc_auc_score(y_val, y_pred_proba)precision, recall, _ = precision_recall_curve(y_val, y_pred_proba)pr_auc = auc(recall, precision)precision_val = precision_score(y_val, y_pred)recall_val = recall_score(y_val, y_pred)f1_val = f1_score(y_val, y_pred)print('='*70)print('ОСНОВНАЯ МЕТРИКА:')print(f'  ROC-AUC: {roc_auc:.4f}')print('='*70)print('\nДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ (для самопроверки):')print(f'  PR-AUC:    {pr_auc:.4f}')print(f'  Precision: {precision_val:.4f}')print(f'  Recall:    {recall_val:.4f}')print(f'  F1-Score:  {f1_val:.4f}')print('='*70)print('\nClassification Report:')print(classification_report(y_val, y_pred, digits=4))

### 8.1 ROC и PR кривые

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))# ROC Curvefpr, tpr, _ = roc_curve(y_val, y_pred_proba)axes[0].plot(fpr, tpr, linewidth=3, label=f'ROC-AUC = {roc_auc:.4f}')axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')axes[0].set_xlabel('False Positive Rate')axes[0].set_ylabel('True Positive Rate')axes[0].set_title('ROC Curve', fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)# PR Curveprecision_curve, recall_curve, _ = precision_recall_curve(y_val, y_pred_proba)axes[1].plot(recall_curve, precision_curve, linewidth=3, label=f'PR-AUC = {pr_auc:.4f}', color='green')axes[1].set_xlabel('Recall')axes[1].set_ylabel('Precision')axes[1].set_title('Precision-Recall Curve', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 9. Предсказания на test set

In [ ]:
final_model_full = LGBMClassifier(**best_params)final_model_full.fit(X, y)test_predictions_proba = final_model_full.predict_proba(test_df)[:, 1]test_predictions = (test_predictions_proba > 0.5).astype(int)submission = pd.DataFrame({'ID': test_ids, 'LoanApproved': test_predictions})submission.to_csv('submission.csv', index=False)print(f'Submission сохранен: {len(submission)} предсказаний')submission.head(10)

## 10. Выводы**Основная метрика:** ROC-AUC ≈ 0.98 (>> 0.75)**Дополнительные метрики:** PR-AUC ≈ 0.98, Precision ≈ 0.93, Recall ≈ 0.93, F1-Score ≈ 0.93**Модель:** LightGBM с оптимизацией через Optuna (50 попыток)**Feature Engineering:** создано 5+ новых признаков